In [1]:
conclusions = {
    "classifier": {
        "best_model_baseline": "cointegrated/rubert-tiny2",
        "best_model_quality": "DeepPavlov/rubert-base-cased",
        "baseline_accuracy": 0.296,
        "baseline_f1": 0.120,
        "quality_accuracy": 0.667,
        "quality_f1": 0.533,
        "note": "Низкие метрики обусловлены размером датасета (~10 примеров/класс). "
                "С датасетом 50+ примеров/класс ожидается Accuracy 0.85+ и F1 0.80+",
        "inference_time_gpu_ms": {
            "rubert-tiny2": 3.6,
            "rubert-base": 29.1,
            "xlm-roberta": 36.9,
            "sbert_large_nlu_ru": 115.4
        },
        "similarity_analysis": {
            "best_without_finetuning": "sbert_large_nlu_ru — лучшее семантическое разделение из коробки",
            "worst_without_finetuning": "rubert-base и xlm-roberta — CLS-токен не различает тексты без fine-tune",
            "rubert_tiny2": "умеренное разделение даже без fine-tune"
        },
        "pros": {
            "rubert-tiny2": [
                "Минимальный размер (111 MB)",
                "Молниеносный инференс (3.6ms GPU)",
                "Лучший для прототипа и демо на защите",
                "Умеренное семантическое разделение без fine-tune"
            ],
            "rubert-base": [
                "Лучшее качество классификации после fine-tune",
                "Проверенная модель для русского NLP",
                "Укладывается в NFR по скорости (29ms)"
            ]
        },
        "cons": {
            "rubert-tiny2": [
                "Ниже качество на малых данных",
                "Может путать семантически близкие категории"
            ],
            "rubert-base": [
                "В 6 раз тяжелее tiny2 (678 MB)",
                "В 8 раз медленнее инференс",
                "CLS-эмбеддинги бесполезны без fine-tuning"
            ]
        },
        "decision": "Стратегия двух моделей: "
                    "1) rubert-tiny2 — baseline и MVP; "
                    "2) rubert-base — основная модель после сбора полного датасета. "
                    "Сравнение двух моделей — сильный аргумент для ВКР."
    },
    "embedder": {
        "best_model": "intfloat/multilingual-e5-small",
        "hit_at_1": "9/10 (90%)",
        "hit_at_3": "10/10 (100%)",
        "avg_similarity": 0.864,
        "dimension": 384,
        "search_time_ms": 24.4,
        "comparison": {
            "MiniLM-L12": {
                "hit_at_1": "7/10 (70%)",
                "avg_similarity": 0.473,
                "verdict": "Similarity слишком низкий для порога 0.70 — непригодна"
            },
            "e5-small": {
                "hit_at_1": "9/10 (90%)",
                "avg_similarity": 0.864,
                "verdict": "Явный победитель — все ответы выше порога 0.70"
            },
            "mpnet-base": {
                "hit_at_1": "9/10 (90%)",
                "avg_similarity": 0.594,
                "verdict": "Hit@1 хороший, но similarity ниже порога"
            }
        },
        "decision": "Выбираю intfloat/multilingual-e5-small: "
                    "лучший Hit@1 (90%), все ответы проходят порог 0.70, "
                    "384-мерные векторы (компактные), быстрый поиск (24ms). "
                    "Единственная модель, работающая с установленным порогом RAG."
    },
    "generator": {
        "best_approach": "template (MVP) + GigaChat/OpenAI (улучшение)",
        "quality_assessment": "Template достаточен для MVP — возвращает текст чанка. "
                              "Для естественности ответов подключить LLM API на следующем этапе.",
        "latency": "Template: <100ms, LLM API: 1-3s",
        "decision": "Начинаем с template (zero cost, zero latency, zero hallucinations). "
                    "Добавляем GigaChat API как улучшение. "
                    "Переключение через конфиг rag_generator_mode."
    },
    "overall": {
        "total_pipeline_time_estimated_ms": {
            "faq_path": "< 50ms (embedder search)",
            "classification_path": "30-120ms (classifier) + 25ms (RAG search) + <100ms (template) = ~250ms",
            "with_llm": "30-120ms + 25ms + 1500ms (API) = ~1700ms"
        },
        "fits_nfr_perf": True,
        "gpu_benefit": "Все модели работают быстро на GPU. "
                       "Даже sbert_large (115ms) укладывается в лимиты. "
                       "CPU тоже приемлем для прототипа.",
        "key_findings": [
            "e5-small — единственный embedder, работающий с порогом similarity 0.70",
            "MiniLM-L12 и mpnet-base требуют снижения порога, что ухудшает precision",
            "rubert-base значительно лучше tiny2 на классификации даже на малых данных",
            "sbert_large_nlu_ru — лучшее семантическое разделение без fine-tune, но самая тяжёлая",
            "Все модели укладываются в NFR по скорости на GPU",
            "Главный bottleneck — размер датасета, а не модель"
        ],
        "next_steps": [
            "1. Собрать реальный датасет: минимум 50 примеров на категорию (500+ всего)",
            "2. Fine-tune rubert-tiny2 и rubert-base на реальных данных",
            "3. Сравнить метрики — зафиксировать для ВКР",
            "4. Поменять VECTOR(384) в БД под e5-small",
            "5. Реализовать FastAPI с выбранными моделями",
            "6. Написать JAICP-сценарии",
            "7. Интегрировать и протестировать"
        ]
    }
}

import json
with open("../results/conclusions.json", "w", encoding="utf-8") as f:
    json.dump(conclusions, f, ensure_ascii=False, indent=2)

print("✅ Выводы сохранены в results/conclusions.json")

✅ Выводы сохранены в results/conclusions.json
